# 17 · Speculative decoding and multi-LoRA
Level L3 · Part 2 · core · ~20 min · builds on 12

## Goal
By the end of this lesson you can explain draft-and-verify decoding and what the acceptance rate does to the speed-up, say when speculation stops helping, and explain how many LoRA adapters share one base model in memory.

## Why it matters
- **Without speculative decoding:** every decode step reads all the weights to make one token. An 8B model in FP16 is about 16 GB of weights, and an L4 reads about 300 GB per second, so one token costs about 53 ms before any other work. At a batch of one, most of the GPU's arithmetic units sit idle while it waits for memory.
- **Without multi-LoRA:** each fine-tuned variant needs its own engine with its own copy of the base weights. Ten language variants of the Brightside model would need about 160 GB of weights.
- **With them:** a cheap guess per step turns idle compute into extra tokens, and one base model serves many adapters in the same batch.
- **When you don't need them:** a busy endpoint with big batches has no idle compute to spend, and one fixed model with no variants needs no adapter.

## The idea
Speculative decoding is the way a fast typist works: they read ahead a few words, and only stop to check the whole phrase at once. A small proposer guesses the next *k* tokens. The large **target model** checks all *k* guesses in one forward pass, keeps the ones it agrees with, and adds one token of its own. Multi-LoRA works the other way round. The base model stays loaded once. Each **LoRA adapter** is a pair of small matrices that adds a correction on top of it, so many adapters can share one copy of the base weights.

## Picture
```mermaid
flowchart LR
  D["Draft: cheap proposer guesses k tokens"] --> V["Target: one forward pass checks all k"]
  V --> A["Keep the agreed prefix, plus one token from the target"]
  A --> D
```

```mermaid
flowchart TB
  B["One base model in GPU memory (about 16 GB for 8B FP16)"]
  B --> L1["Adapter billing-en (about 14 MB)"]
  B --> L2["Adapter billing-de (about 14 MB)"]
  R["Request with model = billing-de"] --> L2
```

## Explained
**Why guesses are cheap to check.** Decode is memory-bound (lesson 06): one step reads every weight to produce one token. Checking *k* guesses reads the weights once and produces *k* positions of output. At a batch of one that costs about the same as one normal step. Standard verification is built so that the kept tokens follow the target model's own distribution, so the answer does not change.

**Four ways to propose the guesses.** A **draft model** is a smaller model that shares the target's tokenizer. **EAGLE** is a small head trained on the target's hidden states. **MTP** uses prediction heads that some models ship with. **N-gram** needs no model: it finds the last few tokens in the prompt and copies what followed them last time. Brightside answers often repeat policy sentences word for word, so n-gram is a cheap first thing to try. The vLLM flag is one JSON setting; the server is not started in this lesson.

```
vllm serve <target-model> --speculative-config '{"method": "ngram", "num_speculative_tokens": 4}'
```

**Acceptance rate and where the gain goes.** The **acceptance rate** *α* is the share of guesses the target keeps. If each guess is kept with probability *α* on its own, the target keeps on average *E = (1 − α^(k+1)) / (1 − α)* tokens per pass. The speed-up is smaller than *E*, because each round also pays for the proposer. The gain is largest at low batch and medium-to-low traffic. At high batch the verify pass does real extra work for every request, and the gain shrinks. Speculation shortens the time between tokens (decode). It does not make the first token faster, because the prompt is still processed the same way (lesson 12).

**How adapters share the base.** A LoRA adapter adds a low-rank correction to a weight matrix *W*: the layer computes *W x + B (A x)*, where *A* is *r × d_in* and *B* is *d_out × r*. The rank *r* is small (8, 16 or 64), so each adapter is a tiny fraction of the base. The engine loads the base once and keeps the adapters it needs. A request names its adapter in the `model` field, and requests for different adapters can share a batch.

`--max-loras` sets how many adapters can be active in one batch, and `--max-lora-rank` sets the largest rank it accepts. Prefix cache entries are tied to the adapter that made them, so one adapter's cache never serves another (lesson 14). Enabling adapters on the server (not started here):

```
vllm serve <base-model> --enable-lora --max-loras 4 --max-lora-rank 16 --lora-modules billing-de=<path-to-adapter>
```

## Worked example: how far the guesses go, and what an adapter costs
Two small calculations with the numbers used above. The first uses four draft tokens per round and a proposer that costs 5% of one target pass per token, a cost model chosen for illustration (not measured). The second sizes the adapters for Llama 3.1 8B (32 layers, hidden size 4,096, KV width 1,024), with rank 16 on the query and value projections, in FP16.

In [1]:
def expected_tokens(alpha, k):
    """Tokens the target keeps per pass when each of k guesses is kept with probability alpha (independent).
    The target always adds one token of its own, which is why the sum runs to k."""
    return (1 - alpha ** (k + 1)) / (1 - alpha)

K, DRAFT_COST = 4, 0.05        # 4 guesses per round; proposer cost per token, as a share of one target pass
print(f"{'alpha':>6} {'tokens per pass':>16} {'speed-up (cost model)':>22}")
for alpha in (0.5, 0.7, 0.8, 0.9):
    e = expected_tokens(alpha, K)
    speedup = e / (K * DRAFT_COST + 1)         # one target pass plus the proposer's work for the round
    print(f"{alpha:>6.1f} {e:>16.2f} {speedup:>21.2f}x")

HIDDEN, KV_WIDTH, LAYERS, RANK, BYTES = 4096, 1024, 32, 16, 2   # Llama 3.1 8B: q and v projections, FP16
def lora_params(d_in, d_out, r):
    return r * (d_in + d_out)                  # A is r x d_in, B is d_out x r
adapter_bytes = (lora_params(HIDDEN, HIDDEN, RANK) + lora_params(HIDDEN, KV_WIDTH, RANK)) * LAYERS * BYTES
base_bytes = 8.0e9 * BYTES                      # about 8B parameters in FP16
print(f"base model: {base_bytes / 1e9:.0f} GB, one rank-{RANK} adapter: {adapter_bytes / 1e6:.1f} MB")
print(f"10 separate engines: {10 * base_bytes / 1e9:.0f} GB of weights")
print(f"1 shared base + 10 adapters: {(base_bytes + 10 * adapter_bytes) / 1e9:.2f} GB")

 alpha  tokens per pass  speed-up (cost model)
   0.5             1.94                  1.61x
   0.7             2.77                  2.31x
   0.8             3.36                  2.80x
   0.9             4.10                  3.41x
base model: 16 GB, one rank-16 adapter: 13.6 MB
10 separate engines: 160 GB of weights
1 shared base + 10 adapters: 16.14 GB


**What you should see:** with an acceptance rate of 0.8, the target keeps about 3.4 tokens per pass and the speed-up under this cost model is about 2.8x. At 0.5 the gain falls to about 1.6x, and it is measured against the proposer's cost, not against zero. The adapter is about 13.6 MB, so ten adapters add about 0.14 GB to one 16 GB base. Ten separate engines would hold about 160 GB. These are counts and arithmetic, not timings: lesson 27 measures real gains.

## At work
- **Cloudflare (May 2026)** lists an EAGLE-3 draft model among the techniques in its production LLM serving stack. Measure the gain on your own traffic before you turn it on for everything. → lesson 27 (benchmarks under load).
- **Speculation helps decode, not the first token.** Production teams split the two problems: a slow first token needs prefix caching and routing, not a draft model. → lesson 14 (prefix caching) and lesson 12 (chunked prefill).
- **A speculation setting is a quality change too.** vLLM's documentation notes that batch size can change logprobs slightly, so exact-match diffs are noisy. Gate the change with a regression set and task metrics. → lesson 21 (measuring quality loss).

Sources: Cloudflare, "High-performance LLMs" (2026-05), blog.cloudflare.com/high-performance-llms/; vLLM documentation, speculative decoding and LoRA pages, docs.vllm.ai (checked 2026-10-21).

## Common misconceptions
1. **"A draft model makes the answer worse, so the output changes."** Standard verification keeps the target's distribution. The draft only proposes. A checked guess is one the target itself would have produced.
2. **"A high acceptance rate always means a big speed-up."** The proposer has a cost, and at a large batch the verify pass costs real compute. The same acceptance rate gives a smaller gain there.
3. **"A LoRA adapter is a second model."** It is a pair of small matrices per adapted layer. At rank 16 on 8B it is about 14 MB, against about 16 GB for the base.
4. **"Speculation speeds up the first token."** It speeds up decode. The prompt is processed the same way, so time to first token is unchanged.

## Check yourself
1. Brightside answers copy policy sentences, so n-gram guesses are accepted 60% of the time with four guesses per round. Using the formula in the worked example, how many tokens does the target keep per pass?
2. Traffic grows, and the batch goes from 2 to 64 requests. Which way does the speed-up move, and why?
3. The complaint is a slow first token, not slow streaming. Will a draft model fix it? What would you try first?
4. Ten language adapters on an 8B FP16 model: how much weight memory with ten separate engines, and with one shared base?
5. A teammate turns on speculation and says quality is unchanged because the answers look fine. What check do you ask for before the change ships?

1. About 2.3 tokens per pass: (1 − 0.6⁵) / 0.4 = 0.92224 / 0.4 = 2.31.
2. It moves down. At batch 2 the GPU has idle compute, so the extra verify tokens are almost free. At batch 64 the GPU is already busy, and the extra tokens cost real work for every request.
3. No. Speculation shortens decode, and the first token is prefill. Look at prefix caching (lesson 14) for the repeated policy prompt, then chunked prefill and routing (lesson 12).
4. About 160 GB with ten engines (10 × 16 GB). About 16.1 GB with one shared base plus ten 14 MB adapters.
5. A regression run on the golden set with task metrics and a tolerance, compared with the current setting at the same sampling settings. Eyeballing a few answers does not catch a small drop (lesson 21).

## Recap
- A draft proposes *k* tokens and the target checks them in one pass. The target's own distribution is kept, and the speed-up depends on the acceptance rate and the batch size.
- Speculation shortens decode, at low batch. It does not change time to first token.
- A LoRA adapter is a small low-rank correction. One base is loaded once, and each request names its adapter in `model`.

**Interview:** "How does speculative decoding speed up generation without changing the output, and when does it stop helping?" Strong answer: a cheap proposer guesses several tokens, the target verifies them in one pass that costs about one normal decode step, and the output keeps the target's distribution. It stops helping at large batch, where the verify tokens cost real compute.